# Detecção e Classificação de Gatos em Fotos com Múltiplos Animais

- **YOLOv8**: Detecta todos os animais na imagem e recorta cada um
- **ResNet50**: Classifica se cada recorte é um gato ou não

In [ ]:
import numpy as np
import os
import glob
from ultralytics import YOLO
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input, decode_predictions
from tensorflow.keras.preprocessing import image as keras_image
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

yolo_model = YOLO("yolov8n.pt")
resnet_model = ResNet50(weights='imagenet')

CAT_LABELS = [
    'tabby', 'tiger_cat', 'persian_cat', 'siamese_cat',
    'egyptian_cat', 'lynx', 'cat', 'domestic_cat'
]

print("Modelos carregados!")

In [ ]:
def classificar_recorte(img_crop):
    """Classifica um recorte de imagem com ResNet50."""
    img_resized = img_crop.resize((224, 224))
    x = keras_image.img_to_array(img_resized)
    x = np.expand_dims(x, axis=0)
    x = preprocess_input(x)
    preds = resnet_model.predict(x, verbose=0)
    decoded = decode_predictions(preds, top=3)[0]
    return decoded

def eh_gato(decoded_preds):
    """Verifica se alguma das top predições é um gato."""
    for _, label, score in decoded_preds:
        if label.lower() in CAT_LABELS:
            return True, label, score
    return False, decoded_preds[0][1], decoded_preds[0][2]

print("Funcoes de classificacao prontas!")

In [ ]:
def detectar_e_classificar(caminho_imagem):
    """Detecta animais com YOLO e classifica cada um com ResNet50."""
    img = Image.open(caminho_imagem).convert("RGB")
    results = yolo_model(caminho_imagem, verbose=False)

    deteccoes = []
    for result in results:
        for box in result.boxes:
            cls_id = int(box.cls[0])
            cls_name = yolo_model.names[cls_id]
            conf = float(box.conf[0])
            x1, y1, x2, y2 = box.xyxy[0].tolist()

            crop = img.crop((x1, y1, x2, y2))
            preds = classificar_recorte(crop)
            is_cat, label, score = eh_gato(preds)

            deteccoes.append({
                "bbox": (x1, y1, x2, y2),
                "yolo_classe": cls_name,
                "yolo_conf": conf,
                "resnet_label": label,
                "resnet_score": score,
                "eh_gato": is_cat
            })

    return img, deteccoes

print("Pipeline de detecao pronto!")

In [ ]:
def mostrar_resultado(caminho_imagem):
    img, deteccoes = detectar_e_classificar(caminho_imagem)

    fig, ax = plt.subplots(1, figsize=(14, 10))
    ax.imshow(img)
    ax.set_title(os.path.basename(caminho_imagem), fontsize=14, fontweight='bold')

    gatos_encontrados = 0

    for det in deteccoes:
        x1, y1, x2, y2 = det["bbox"]
        w, h = x2 - x1, y2 - y1

        if det["eh_gato"]:
            cor = "lime"
            texto = f"GATO ({det['resnet_label']}) {det['resnet_score']*100:.1f}%"
            gatos_encontrados += 1
        else:
            cor = "red"
            texto = f"{det['yolo_classe']} -> {det['resnet_label']} {det['resnet_score']*100:.1f}%"

        rect = patches.Rectangle((x1, y1), w, h, linewidth=3, edgecolor=cor, facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1 - 8, texto, color='white', fontsize=10, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor=cor, alpha=0.8))

    total = len(deteccoes)
    resumo = f"Animais detectados: {total} | Gatos: {gatos_encontrados} | Outros: {total - gatos_encontrados}"
    ax.text(0.5, -0.05, resumo, transform=ax.transAxes, fontsize=12, fontweight='bold',
            ha='center', color='white',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='black', alpha=0.8))

    ax.axis('off')
    plt.tight_layout()
    plt.show()

    return deteccoes

print("Visualizacao pronta!")

In [ ]:
ASSETS_DIR = "assets"
EXTENSOES = ("*.png", "*.jpg", "*.jpeg", "*.webp")

arquivos = []
for ext in EXTENSOES:
    arquivos.extend(glob.glob(os.path.join(ASSETS_DIR, ext)))
arquivos.sort()

print(f"Imagens encontradas: {len(arquivos)}")
for f in arquivos:
    print(f"  - {os.path.basename(f)}")

todos_resultados = []
for foto in arquivos:
    print(f"\n{'='*60}")
    print(f"Analisando: {os.path.basename(foto)}")
    print(f"{'='*60}")
    deteccoes = mostrar_resultado(foto)
    todos_resultados.append({"imagem": os.path.basename(foto), "deteccoes": deteccoes})

In [ ]:
import pandas as pd

linhas = []
for r in todos_resultados:
    for d in r["deteccoes"]:
        linhas.append({
            "Imagem": r["imagem"],
            "YOLO Classe": d["yolo_classe"],
            "YOLO Conf": f"{d['yolo_conf']*100:.1f}%",
            "ResNet Label": d["resnet_label"],
            "ResNet Score": f"{d['resnet_score']*100:.1f}%",
            "Gato?": "Sim" if d["eh_gato"] else "Nao"
        })

df = pd.DataFrame(linhas)
display(df)

total_animais = len(df)
total_gatos = len(df[df["Gato?"] == "Sim"])
print(f"\nTotal de animais detectados: {total_animais}")
print(f"Total de gatos identificados: {total_gatos}")
print(f"Total de outros animais: {total_animais - total_gatos}")